# Manual resolution walkthrough

This notebook is documentation, not part of the production pipeline: it shows how the three
hand-curated CSVs in `data/manual_resolutions/` turn into real `sead_staging` ids, by calling the
exact same `shared.resolution.species`/`shared.resolution.material` functions
`c14_v1_dataset_transformation.ipynb` uses. Nothing here is a required input to that pipeline -
this is purely for inspecting/sanity-checking the resolution step in isolation.

## Where the three manual CSVs come from

- **`species_manual_resolution.csv`** started as the automated species/GBIF match built in
  `archive/notebooks/species_study.ipynb`, was hand-corrected into
  `archive/data/species_split_counts_in_original_manual_v4.csv` (typos fixed, duplicates merged),
  re-matched against SEAD + GBIF in `archive/notebooks/species_split_for_study_etl.ipynb`, and
  finally hand-completed (every row given a resolved order/family/genus/species and a Swedish
  common name) into what's now `species_manual_resolution.csv`. See `archive/output/species/plan.md`
  for the original design notes behind that re-matching pass.
- **`material_manual_resolution.csv`** started as raw value counts from
  `archive/notebooks/material_counts.ipynb`, then was hand-broken down into up to 3 SEAD elements
  + an optional modification, per distinct raw `material` value.
- **`species_token_corrections.csv`** is the `species_split` -> `manual_species` correction table
  used to reconcile whichever raw dataset's split species tokens (v08's or v1's) against the
  vocabulary the two files above were built from.

None of that re-matching/re-melting work is repeated here - these three files are already the
finished hand-curated decisions. What *is* live below is turning those decisions into real SEAD
ids, since that step must always run fresh against the current DB state.


In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('../..').resolve()))
from shared.resolution import common, species as species_resolution, material as material_resolution

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# .env lives at the true repo root, two levels up from current/notebooks/.
engine = common.get_db_engine('../../.env')


## Species resolution


In [2]:
species_manual_df = pd.read_csv(
    '../data/manual_resolutions/species_manual_resolution.csv', keep_default_na=False, na_values=[''],
)
print(f'{len(species_manual_df)} manually-resolved species rows loaded')
species_manual_df[
    ['manual_species', 'resolved_order', 'resolved_family', 'resolved_genus', 'resolved_species',
     'common_name_text', 'needs_manual_review']
].head(10)


212 manually-resolved species rows loaded


,manual_species,resolved_order,resolved_family,resolved_genus,resolved_species,common_name_text,needs_manual_review
0,al,Fagales,Betulaceae,Alnus,sp.,al,True
1,tall,Pinales,Pinaceae,Pinus,sylvestris var sylvestris,tall,False
2,korn,Poales,Poaceae,Hordeum,vulgare,korn,True
3,björk,Fagales,Betulaceae,Betula,sp.,björk,False
4,ek,Fagales,Fagaceae,Quercus,robur,ek,False
5,ko,Artiodactyla,Bovidae,Bos,taurus,ko,False
6,hassel,Fagales,Corylaceae,Corylus,avellana,hassel,False
7,gran,Pinales,Pinaceae,Picea,abies ssp abies,gran,False
8,ran,Pinales,Pinaceae,Picea,abies ssp abies,gran,True
9,människa,Primates,Hominidae,Homo,sapiens,människa,False


In [3]:
resolved_species, new_species_records = species_resolution.resolve_species_ids(species_manual_df, engine)

n_resolved = resolved_species['taxon_id'].notna().sum()
n_new = int(resolved_species['taxon_id_is_new'].sum())
n_blocked = int(resolved_species['blocked_by_existing_author_id'].sum())
print(f'{n_resolved} of {len(resolved_species)} rows resolved to a taxon_id')
print(f'  {n_new} of those are newly-proposed taxon_ids (no matching SEAD taxon exists yet)')
print(f'  {n_resolved - n_new} matched an existing SEAD taxon')
print(f'  {n_blocked} rows had a name match blocked by an existing author_id (species text matched, '
      f'but every matching SEAD row has a non-null author_id, so a new taxon_id was proposed instead '
      f'of reusing one that could carry an unintended taxonomic-authority attribution)')

resolved_species[
    ['manual_species', 'resolved_order', 'resolved_family', 'resolved_genus', 'resolved_species',
     'taxon_id', 'taxon_id_is_new', 'common_name_id', 'common_name_id_is_new']
].head(10)


212 of 212 rows resolved to a taxon_id
  139 of those are newly-proposed taxon_ids (no matching SEAD taxon exists yet)
  73 matched an existing SEAD taxon
  45 rows had a name match blocked by an existing author_id (species text matched, but every matching SEAD row has a non-null author_id, so a new taxon_id was proposed instead of reusing one that could carry an unintended taxonomic-authority attribution)


,manual_species,resolved_order,resolved_family,resolved_genus,resolved_species,taxon_id,taxon_id_is_new,common_name_id,common_name_id_is_new
0,al,Fagales,Betulaceae,Alnus,sp.,18087,False,4273.0,True
1,tall,Pinales,Pinaceae,Pinus,sylvestris var sylvestris,3613,False,2797.0,False
2,korn,Poales,Poaceae,Hordeum,vulgare,18010,False,4274.0,True
3,björk,Fagales,Betulaceae,Betula,sp.,18086,False,4275.0,True
4,ek,Fagales,Fagaceae,Quercus,robur,47010,True,4276.0,True
5,ko,Artiodactyla,Bovidae,Bos,taurus,47011,True,4277.0,True
6,hassel,Fagales,Corylaceae,Corylus,avellana,47012,True,4278.0,True
7,gran,Pinales,Pinaceae,Picea,abies ssp abies,3591,False,2782.0,False
8,ran,Pinales,Pinaceae,Picea,abies ssp abies,3591,False,2782.0,False
9,människa,Primates,Hominidae,Homo,sapiens,47013,True,4279.0,True


`new_species_records` lists every order/family/genus/taxon/common_name row this run would need to
propose to SEAD - a to-do list for whoever eventually runs the real INSERTs. No INSERTs happen
here; the DB connection this notebook uses is read-only.


In [4]:
print(f'{len(new_species_records)} new SEAD records proposed')
new_species_records['table'].value_counts()


306 new SEAD records proposed


table
tbl_taxa_common_names     139
tbl_taxa_tree_master       92
tbl_taxa_tree_genera       35
tbl_taxa_tree_families     24
tbl_taxa_tree_orders       16
Name: count, dtype: int64

## Material resolution


In [5]:
material_manual_df = pd.read_csv(
    '../data/manual_resolutions/material_manual_resolution.csv', keep_default_na=False, na_values=[''],
)
print(f'{len(material_manual_df)} manually-resolved material rows loaded')
material_manual_df[
    ['material', 'sead_element_1', 'sead_element_2', 'sead_element_3', 'sead_modification_type',
     'sead_record_type_id']
].head(10)


42 manually-resolved material rows loaded


,material,sead_element_1,sead_element_2,sead_element_3,sead_modification_type,sead_record_type_id
0,Träkol,Charcoal,NaN,NaN,NaN,9
1,Förkolnat frö,Seed,NaN,NaN,Carbonised,2
2,"Ben, obrända",Bone(s),NaN,NaN,NaN,14
3,"Ben, brända",Bone(s),NaN,NaN,Carbonised,14
4,Trä,Wood,NaN,NaN,NaN,2
5,Skalfragment,Shell,NaN,NaN,NaN,2
6,Organisk beläggning,Unknown,NaN,NaN,NaN,0
7,Växtdelar,Unspecified part,NaN,NaN,NaN,2
8,Bark,Bark,NaN,NaN,NaN,2
9,Harts,Resin,NaN,NaN,NaN,2


In [6]:
resolved_material, new_material_records = material_resolution.resolve_material_ids(material_manual_df, engine)

n_new_elements = sum(resolved_material[f'sead_element_{i}_is_new'].sum() for i in (1, 2, 3))
n_new_mods = int(resolved_material['modification_type_is_new'].sum())
print(f'{int(n_new_elements)} element assignments are newly proposed, {n_new_mods} modification-type '
      f'assignments are newly proposed')

resolved_material[
    ['material', 'sead_element_1', 'sead_element_1_id', 'sead_element_1_is_new',
     'sead_modification_type', 'sead_modification_type_id', 'sead_record_type_id']
].head(10)


37 element assignments are newly proposed, 0 modification-type assignments are newly proposed


,material,sead_element_1,sead_element_1_id,sead_element_1_is_new,sead_modification_type,sead_modification_type_id,sead_record_type_id
0,Träkol,Charcoal,50,True,NaN,NaN,9
1,Förkolnat frö,Seed grain,4,False,Carbonised,1.0,2
2,"Ben, obrända",Bone(s),51,True,NaN,NaN,14
3,"Ben, brända",Bone(s),51,True,Carbonised,1.0,14
4,Trä,Wood,44,False,NaN,NaN,2
5,Skalfragment,Shell,39,False,NaN,NaN,2
6,Organisk beläggning,Unknown,52,True,NaN,NaN,0
7,Växtdelar,Unspecified part,53,True,NaN,NaN,2
8,Bark,Bark,43,False,NaN,NaN,2
9,Harts,Resin,54,True,NaN,NaN,2


In [7]:
print(f'{len(new_material_records)} new SEAD records proposed')
new_material_records['table'].value_counts()


20 new SEAD records proposed


table
tbl_abundance_elements    20
Name: count, dtype: int64